# pyobs-indi: is the epoch conversion right?

INDI reports `EQUATORIAL_EOD_COORD` in **equinox of date**, with right ascension in
**hours**. pyobs and the Stellarium protocol are **J2000 degrees**. The module converts
in both directions, and a wrong conversion fails quietly -- it produces a plausible
coordinate rather than an error.

So every slew is logged, and this is the picture that would show it.

**What to look for.** A fixed rounding in right ascension covers less sky as declination
rises, by `cos(dec)` -- so arrival error should *fall* toward the pole. If the `tan(dec)`
term in the conversion were wrong, the points would climb away from that curve instead:
right at the equator, hopeless at the pole.

The curve is scaled on the low-declination points only, so the pole is a prediction
rather than something the fit was handed.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
from plot_log import load, plot_error_vs_dec

df = load()
print(f"{len(df)} arrivals, declination {df.dec_j2000.min():+.1f} to {df.dec_j2000.max():+.1f}")
df.tail()

In [ ]:
%matplotlib inline
plot_error_vs_dec(df);

## Slew duration

How long each slew took, against how far it had to travel. Straight line means constant
slew rate; a knee would mean acceleration limits or a settle time that does not scale.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

raw = pd.read_csv('indi-log.csv', parse_dates=['time'])
slews = raw[raw.event == 'slew'].reset_index(drop=True)
arr   = raw[raw.event == 'arrived'].reset_index(drop=True)
n = min(len(slews), len(arr))
secs = (arr.time[:n].values - slews.time[:n].values) / np.timedelta64(1, 's')

# angular distance from the previous target to this one
from astropy.coordinates import SkyCoord
import astropy.units as u
c = SkyCoord(ra=slews.ra_j2000[:n].values*u.deg, dec=slews.dec_j2000[:n].values*u.deg)
dist = np.concatenate([[np.nan], c[1:].separation(c[:-1]).deg])

plt.figure(figsize=(8,5), dpi=160)
plt.scatter(dist, secs, s=55, color='#1f4e79')
plt.xlabel('distance from previous target (degrees)')
plt.ylabel('slew duration (seconds)')
plt.title('pyobs-indi: slew duration vs distance')
plt.grid(alpha=.3); plt.tight_layout()